## PROJET PARIS TRAFFIC Part 2


**Contexte** : Cette partie va nous permettre de nettoyer les données et de justifier chaque choix avec tes chiffres. Tout d'abord on va telecharger les données.

**Telechargement des données**

In [1]:
import os 
import pandas as pd
import matplotlib.pyplot as plt
CHEMIN = "data/raw/comptages.csv"

**Le chargement par chunks**

In [2]:
TYPES = {
    "iu_ac": "string", "iu_nd_amont": "string", "iu_nd_aval": "string",
    "libelle": "category", "etat_trafic": "category", "etat_barre": "category",
}

morceaux = []
for chunk in pd.read_csv(CHEMIN, sep=";", encoding="utf-8-sig",
                         dtype=TYPES, chunksize=50_000):
    morceaux.append(chunk)

df = pd.concat(morceaux, ignore_index=True)
df.shape

(95960, 11)

## 1. Dates en datetime et en heure de Paris, identifiants en texte, états en category

*a/ Dates en datetime et en heure de Paris* 

Regardons d'abord à quoi ressemble t_1h :

In [3]:
print(df["t_1h"].dtype, df["t_1h"].iloc[0])
df["t_1h"].head(3)


str 2026-08-31T23:00:00+00:00


0    2026-08-31T23:00:00+00:00
1    2026-08-31T23:00:00+00:00
2    2026-08-31T23:00:00+00:00
Name: t_1h, dtype: str

In [4]:
df["t_1h"].str.len().value_counts()
df["t_1h"].str[-6:].value_counts()

t_1h
+00:00    95960
Name: count, dtype: int64

t_1h est du texte (format str) , toutes les lignes finissent par +00:00 sans exception donc on a pas de format mélanger , ils ont le format AAAA-MM-JJTHH:MM:SS+00:00.

Puis la conversion

In [5]:
df["t_utc"] = pd.to_datetime(df["t_1h"], utc=True)
df["t_paris"] = df["t_utc"].dt.tz_convert("Europe/Paris")
df[["t_1h", "t_utc", "t_paris"]].head(3)

,t_1h,t_utc,t_paris
0,2026-08-31T23:00:00+00:00,2026-08-31 23:00:00+00:00,2026-09-01 01:00:00+02:00
1,2026-08-31T23:00:00+00:00,2026-08-31 23:00:00+00:00,2026-09-01 01:00:00+02:00
2,2026-08-31T23:00:00+00:00,2026-08-31 23:00:00+00:00,2026-09-01 01:00:00+02:00


In [6]:
print(df["t_utc"].dtype)
print(df["t_paris"].dtype)
print("Dates non converties :", df["t_utc"].isna().sum())

datetime64[us, UTC]
datetime64[us, Europe/Paris]
Dates non converties : 0


Vérifions que mes données couvrent bien la période du 1er mars au 31 août sans trou au début ni à la fin. Je regarde aussi l'effet de la conversion sur les bords de la période.

In [7]:
print("UTC   :", df["t_utc"].min(), "->", df["t_utc"].max())
print("Paris :", df["t_paris"].min(), "->", df["t_paris"].max())

UTC   : 2026-03-01 00:00:00+00:00 -> 2026-08-31 23:00:00+00:00
Paris : 2026-03-01 01:00:00+01:00 -> 2026-09-01 01:00:00+02:00


En UTC, du 01/03 00h au 31/08 23h : le fichier couvre exactement la période.
Pour Paris du 01/03 01h (+01:00) au 01/09 01h (+02:00) : la fenêtre est décalée de 1 h au début et de 2 h à la fin. Le passage de +01:00 à +02:00 montre aussi que le changement d'heure est pris en compte.

L'étiquette `+00:00` affirme que les heures sont en UTC, mais une étiquette peut être fausse. Si je convertis sans vérifier, toutes mes heures seraient décalées d'1 ou 2 h, et mes heures de pointe seraient fausses. Je teste donc avec un jour où l'heure change : le 29 mars 2026, Paris passe de 02h à 03h, donc cette journée ne dure que 23 h en heure locale.

In [8]:
un_capteur = df[df["iu_ac"] == df["iu_ac"].iloc[0]]

for jour in ["2026-03-28", "2026-03-29", "2026-03-30"]:
    n_utc = (un_capteur["t_utc"].dt.strftime("%Y-%m-%d") == jour).sum()
    n_paris = (un_capteur["t_paris"].dt.strftime("%Y-%m-%d") == jour).sum()
    print(jour, "| UTC :", n_utc, "heures | Paris :", n_paris, "heures")

2026-03-28 | UTC : 24 heures | Paris : 24 heures
2026-03-29 | UTC : 24 heures | Paris : 23 heures
2026-03-30 | UTC : 24 heures | Paris : 24 heures


On a UTC : 24/ 24/24 donc la série horaire n'a aucune heure manquante, y compris le 29 mars. Puis pour Paris 24/23/24 la journée du 29 mars compte 23 heures en heure locale donc que tz_convert gère le changement d'heure. Tout est ok!

Verifions maintenant sur toute la période que le décalage passe bien de +1 h (hiver) à +2 h (été), et pas seulement le 29 mars.

In [9]:
df["decalage"] = df["t_paris"].dt.strftime("%z")
df.groupby(df["t_paris"].dt.month)["decalage"].agg(lambda s: s.unique().tolist())

t_paris
3    [+0200, +0100]
4           [+0200]
5           [+0200]
6           [+0200]
7           [+0200]
8           [+0200]
9           [+0200]
Name: decalage, dtype: object

Le passage de l'heure d'été à bien été appliqué sur toute la période comme on voulait!

In [10]:
df = df.drop(columns="decalage") #pour ne pas encombrer le dataframe avec une colonne temporaire

In [ ]:
df = df.drop(columns="t_1h") #inutile maintenant que l'on a t_utc et t_paris

*b/ Identifiants en texte et états en `category`*


In [12]:
df.dtypes

iu_ac                                     string
libelle                                 category
q                                        float64
k                                        float64
etat_trafic                             category
iu_nd_amont                               string
libelle_nd_amont                             str
iu_nd_aval                                string
libelle_nd_aval                              str
etat_barre                              category
t_utc                        datetime64[us, UTC]
t_paris             datetime64[us, Europe/Paris]
dtype: object

On voit que on a les bon type.

## 2. Les doublons

In [19]:
print("Lignes entièrement identiques :", df.duplicated().sum())
print("meme capteure, meme heure, meme etat du trafic:", df.duplicated(subset=["iu_ac", "t_paris", "etat_trafic"]).sum())

Lignes entièrement identiques : 0
meme capteure, meme heure, meme etat du trafic: 0


**Test sur la bonne clé**

Une mesure est identifiée par un tronçon et une heure. Deux lignes avec le même `iu_ac` et la même heure décrivent la même mesure. Le test précédent ne les verrait pas ne les verrais pas donc.

In [20]:
cle = ["iu_ac", "t_utc"]
print("Doublons sur la clé (tronçon, heure) :", df.duplicated(subset=cle).sum())

Doublons sur la clé (tronçon, heure) : 0


Jusque là , il n'y a pas de ligne répétée, mais si deux capteurs voisins peuvent renvoyer exactement la même série de mesures les compter séparément donnerait trop de poids à cette information. Vérifions si c'est le cas :

**Les doublons cachés**

In [22]:
def groupes_identiques(df, col): #cette fonction permet de regrouper les capteurs qui ont exactement les mêmes mesures sur la période étudiée
    large = df.pivot_table(index="t_utc", columns="iu_ac", values=col) 
    signature = large.round(3).fillna(-1).T.apply(lambda s: hash(tuple(s)), axis=1) #pour chaque capteur, on calcule un hash de sa série de mesures (après arrondi à 3 décimales et remplacement des NaN par -1)
    return signature.map({h: i for i, h in enumerate(signature.unique())}) #ici on remplace le hash par un numéro de série unique pour chaque groupe de capteurs identiques

groupe_q = groupes_identiques(df, "q") 
groupe_k = groupes_identiques(df, "k")
print(f"Débit q      : {len(groupe_q)} capteurs, {groupe_q.nunique()} séries différentes") 
print(f"Occupation k : {len(groupe_k)} capteurs, {groupe_k.nunique()} séries différentes")

Débit q      : 22 capteurs, 13 séries différentes
Occupation k : 22 capteurs, 20 séries différentes


On a que pour les deux , le nombre de séries est plus petit au nombre de capteurs, donc certains capteurs partagent la même série. 
1-Débit q : Plusieurs capteurs ont exactement la même série de débit, heure par heure. Si les 22 capteurs étaient indépendants, j'aurais 22 séries.
2-Occupation k : 22 capteurs, 20 séries. Donc presque tous les capteurs ont leur propre série. Seuls quelques-uns en partagent.


Maintenant regardons quels capteurs partagent la même série de débit, de quels axes ils viennent, et s'ils sont voisins. Si ce sont des tronçons consécutifs, cela suggère que le débit d'un capteur est recopié sur ses voisins.

In [23]:
# Taille de chaque groupe de débit
print(groupe_q.value_counts())

4     4
5     3
8     3
3     2
6     2
0     1
1     1
2     1
7     1
9     1
10    1
11    1
12    1
Name: count, dtype: int64


In [24]:
# Le plus grand groupe : quels capteurs ?
plus_grand = groupe_q.value_counts().index[0]
arcs = groupe_q[groupe_q == plus_grand].index
df[df["iu_ac"].isin(arcs)].drop_duplicates("iu_ac")[
    ["iu_ac", "libelle", "libelle_nd_amont", "libelle_nd_aval"]
]

,iu_ac,libelle,libelle_nd_amont,libelle_nd_aval
9,1634,Bd_Barbes,Bd_Barbes-Simart-Labat,Bd_Barbes-Marcadet
12,1630,Bd_Barbes,BdBarbes-Custine-Poulet_PlChatR,Bd_Barbes-Doudeauville
14,1632,Bd_Barbes,Bd_Barbes-Doudeauville,Bd_Barbes-Simart-Labat
17,1636,Bd_Barbes,Bd_Barbes-Marcadet,Ornano-Ordener-Barbes-Boinod


Test de fiabilité pour voir si ces groupes ne sont pas des capteurs vides

In [25]:
large_q = df.pivot_table(index="t_utc", columns="iu_ac", values="q")
resume = pd.DataFrame({
    "groupe": groupe_q,
    "axe": df.drop_duplicates("iu_ac").set_index("iu_ac")["libelle"],
    "part_mesures_presentes_%": (large_q.notna().mean() * 100).round(1),
})
resume.sort_values("groupe")

,groupe,axe,part_mesures_presentes_%
iu_ac,,,
1626,0,Bd_Barbes,90.2
1627,1,Bd_Barbes,90.4
1628,2,Bd_Barbes,90.4
1629,3,Bd_Barbes,98.9
1631,3,Bd_Barbes,98.9
1630,4,Bd_Barbes,97.5
1632,4,Bd_Barbes,97.5
1634,4,Bd_Barbes,97.5
1636,4,Bd_Barbes,97.5


**Bilan**

On a 5 groupes qui contiennent plusieurs capteurs (4, 3, 3, 2 et 2), soit 14 capteurs sur 22. Les 8 autres capteurs ont une série à eux.

Par axe :

Axe	            /   Capteurs    	/ Séries de débit différentes

Bd_Barbes	         /  12	   /  6

Av_Pdt_Kennedy     	  / 6	       /  3

Av_Grande_Armee	     /  4	      /  4

l'axe Av_Grande_Armee n'a aucun doublon caché. Les deux autres axes en ont beaucoup : sur Bd_Barbes, 12 capteurs ne donnent que 6 séries.
Par ailleurs ce ne sont pas des capteurs vides , les groupes partagés ont entre 94,6 % et 99,8 % de mesures présentes. 

Ce sont donc de vraies séries recopiées.
Je garde tous les tronçons, car leur occupation est presque toujours propre à chacun, et je conserve la colonne `groupe_q` pour ne compter chaque série de débit qu'une fois. 


Je garde le numéro de groupe de chaque capteur dans le tableau, pour savoir plus tard quels capteurs partagent la même série de débit. Je ne supprime aucune ligne.

In [26]:
df["groupe_q"] = df["iu_ac"].map(groupe_q)
(df.drop_duplicates("iu_ac")
   .groupby("libelle", observed=True)
   .agg(capteurs=("iu_ac", "nunique"), series_q=("groupe_q", "nunique")))

,capteurs,series_q
libelle,,
Av_Grande_Armee,4,4
Av_Pdt_Kennedy,6,3
Bd_Barbes,12,6


## 3.Colonnes inutiles

Faisons d'abord l'inventaire de ce que contient chaque colonne et du nombre de valeurs différentes qu'elle prend. Une colonne qui ne prend qu'une seule valeur n'apporte aucune information.

In [27]:
df.nunique(dropna=False).sort_values()

libelle                 3
etat_barre              3
etat_trafic             5
groupe_q               13
iu_nd_aval             15
libelle_nd_aval        15
libelle_nd_amont       16
iu_nd_amont            16
iu_ac                  22
q                    1389
t_paris              4364
t_utc                4364
k                   38589
dtype: int64

In [ ]:
df.groupby("iu_ac")[["iu_nd_amont", "iu_nd_aval"]].nunique().max() #pour savoir si un capteur peut être associé à plusieurs noeuds amont ou aval

iu_nd_amont    1
iu_nd_aval     1
dtype: int64

In [ ]:
n = df.groupby("iu_ac")["iu_nd_aval"].nunique() 
n[n > 1]

Series([], Name: iu_nd_aval, dtype: int64)

In [30]:
print("t_utc = t_paris :", (df["t_utc"] == df["t_paris"]).all())

t_utc = t_paris : True


In [31]:
a_supprimer = ["t_utc", "iu_nd_amont", "libelle_nd_amont", "iu_nd_aval", "libelle_nd_aval"]
df = df.drop(columns=a_supprimer)
df.columns.tolist()

['iu_ac',
 'libelle',
 'q',
 'k',
 'etat_trafic',
 'etat_barre',
 't_paris',
 'groupe_q']

**Bilan**

Chaque capteur a 1 carrefour amont et au maximum 1 carrefours aval sur la période. `t_utc` et `t_paris` désignent le même instant . 
**Conclusion** : les colonnes de carrefours décrivent la géométrie du réseau, et `iu_ac` suffit à identifier un tronçon, je peux donc  les supprimer. Je garde `t_paris` pour les analyses horaires et supprime `t_utc`, qui contient la même information.

## Valeurs manquantes

In [32]:
df[["q", "k"]].describe()

,q,k
count,92486.000000,86791.000000
mean,505.568896,9.768561
std,257.419681,11.674823
min,0.000000,0.000000
25%,334.000000,2.788890
50%,463.000000,6.277220
75%,634.000000,11.396945
max,5421.950000,97.281110


In [35]:
df[["q", "k"]].isna().mean().round(3) * 100

q    3.6
k    9.6
dtype: float64

In [37]:
pd.crosstab(df["etat_barre"], df["q"].isna(), margins=True)

q,False,True,All
etat_barre,,,
Barré,100,0,100
Invalide,0,3202,3202
Ouvert,92386,272,92658
All,92486,3474,95960
